<a href="https://colab.research.google.com/github/sjanwalkar/MedAssist-AI/blob/main/MedAssist_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Downlaod Libraries
!pip install langchain-community neo4j langchain-text-splitters langchain-chroma langchain_ollama

In [ ]:
!pip install langchain-google-genai langgraph langchain langsmith

In [ ]:
! pip install pypdf

In [ ]:
'''
🎯 Project Overview
Problem It Solves:
Healthcare professionals spend 30-40% of their time on documentation instead of patient care.
MedAssist AI is an intelligent assistant that helps doctors by:

Answering questions about patient medical history instantly
Retrieving relevant medical guidelines and drug interactions
Summarizing previous visits and treatments
Suggesting diagnosis codes based on symptoms

Why This Project Stands Out:
This project demonstrates almost every modern AI/ML skill employers want:
RAG, agents, vector databases, knowledge graphs, embeddings, chunking strategies, and LLM orchestration.
'''

In [ ]:
'''
🏗️ Architecture Overview
User Query → Agent Orchestrator → [Multiple Specialized Tools] → Response Generator
                                    ↓
                    1. Vector Search Tool (RAG)
                    2. Knowledge Graph Tool
                    3. Web Search Tool
                    4. SQL Database Tool

'''

In [ ]:
# Download and explore MTSamples
import pandas as pd
import os

# Load data
df = pd.read_csv('mtsamples.csv')

# Filter for specific specialties relevant to your project
specialties = [' Cardiovascular / Pulmonary', ' Neurology',
               ' Emergency Room Reports', ' Discharge Summary']
filtered_df = df[df['medical_specialty'].isin(specialties)]

# Save as individual text files
os.makedirs("medical_docs", exist_ok=True)
for idx, row in filtered_df.iterrows():
    with open(f"medical_docs/doc_{idx}.txt", 'w') as f:
        f.write(f"SPECIALTY: {row['medical_specialty']}\n\n")
        f.write(f"DESCRIPTION: {row['description']}\n\n")
        f.write(f"TRANSCRIPTION:\n{row['transcription']}")

print(f"Created {len(filtered_df)} medical documents!")

In [ ]:
import requests

# Direct PDF URLs (working examples)
pdf_urls = {
    "diabetes_guideline": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7138134/pdf/main.pdf",
    "hypertension_treatment": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6389732/pdf/main.pdf",
    "covid_management": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7102591/pdf/main.pdf",
    "heart_failure": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6500486/pdf/main.pdf",
    "asthma_guidelines": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6688643/pdf/main.pdf"
}

# Download all PDFs
os.makedirs("medical_pdfs", exist_ok=True)
for name, url in pdf_urls.items():
    print(f"Downloading {name}...")
    response = requests.get(url)

    if response.status_code == 200:
      with open(f"medical_pdfs/{name}.pdf", "wb") as f:
        f.write(response.content)
      print(f"✅ Downloaded {name}.pdf")
    else:
        print(f"❌ Failed to download {name}")

In [ ]:
'''
📊 Detailed Architecture & Flow
Stage 1: Data Ingestion & Preprocessing
What Happens: Raw medical documents (patient records, medical guidelines, research papers) are processed and stored in multiple formats.
Components:

Document Loaders

Load PDFs, Word docs, HTML pages
Extract text while preserving structure
'''
# Document Loader Library
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "medical_pdfs/",
    glob="**/*guidelines.pdf",
    loader_cls=PyPDFLoader,
    silent_errors=True, # Prevents the Traceback from crashing your script
    show_progress=True
)

all_medical_docs = dir_loader.load()
print(f"Total pages loaded from directory: {len(all_medical_docs)}")

# Load ALL .txt files from a directory
loader = DirectoryLoader(
    "medical_docs/",           # folder path
    glob="*.txt",              # load all .txt files
    loader_cls=TextLoader     # use TextLoader for each file
)
text_documents = loader.load()
print(f"Loaded {len(text_documents)} text Documents")

# Output: [Document(page_content="Diabetes treatment...", metadata={page: 1})]

In [ ]:
'''
Chunking Strategy (CRITICAL for good RAG)
Why Chunk? LLMs have token limits, and smaller chunks = more precise retrieval
Strategies Used:

Semantic Chunking: Split by medical topics (symptoms, treatment, diagnosis)
Recursive Chunking: Split large text into 500-token chunks with 50-token overlap
Metadata Enhancement: Add document source, date, patient ID
'''
# Importing Library
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
       chunk_size=500,  # characters per chunk
       chunk_overlap=50,  # overlap to maintain context
       separators=["\n\n"]
   )

chunks = splitter.split_documents(text_documents)
# Before: 1 document (5000 chars)
# After: 10 chunks (500 chars each with overlap)


In [ ]:
'''
Embedding Generation
What's an Embedding? A numerical representation of text (vector) that captures meaning
'''
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# embeddings_model = OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     openai_api_key="sk-your-key-here"
#     )

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Convert chunk to vector
vector = embeddings_model.embed_query("Patient has diabetes")
# Returns: array of 1536 numbers

In [ ]:
'''
Vector Store (for similarity search)
Popular Options:

Pinecone (cloud, easy to use)
ChromaDB (free, local)
Weaviate (open-source, feature-rich)
'''
# Simple example with ChromaDB
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
       documents=chunks,
       embedding=embeddings_model,
       persist_directory="./medical_db"
   )

# Now you can search!
results = vectorstore.similarity_search("diabetes treatment", k=3)
# Returns: 3 most similar chunks

In [ ]:
results = vectorstore.similarity_search("What treatment is John ungergoing?", k=3)

In [ ]:
'''
Knowledge Graph Creation
Why Knowledge Graph? Shows relationships (Patient → has → Disease → requires → Medication)
Tool: Neo4j (graph database)
'''
# Simple example: Storing relationships
   # Patient "John" → "has" → "Diabetes"
   # "Diabetes" → "treated_with" → "Metformin"
   # "Metformin" → "interacts_with" → "Alcohol"

from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687")

def run_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters)
        return [record.data() for record in result]

# Create relationship

import hashlib

def create_patient_record(transcript_text):
    """
    Creates an anonymous patient record in Neo4j using a hash of
    the transcript to ensure uniqueness without names.
    """
    # 1. Generate a unique ID from the transcript text itself
    # This ensures that identical transcripts don't create duplicate patients
    patient_id = "PAT_" + hashlib.md5(transcript_text.encode()).hexdigest()[:8]

    # 2. Advanced Cypher Query for mtSamples logic
    query = """
    // Use MERGE to create a unique anonymous patient
    MERGE (p:Patient {id: $id})

    // Create/Connect the primary condition mentioned
    MERGE (d:Condition {name: $condition})
    MERGE (p)-[:DIAGNOSED_WITH]->(d)

    // Link the plan/medication change mentioned in the transcript
    MERGE (m:Medication {name: $plan_med})
    MERGE (p)-[:PLAN_TO_TRY]->(m)

    RETURN p.id AS pid, d.name AS cond, m.name AS med
    """

    # In a full LangChain agent, these values would be extracted by an LLM node
    # Here we simulate the extraction for the Allergy transcript provided:
    params = {
        "id": patient_id,
        "condition": "Allergic rhinitis",
        "plan_med": "Zyrtec"
    }

    res = run_query(query, params)

    if res:
        record = res[0]
        return f"Created anonymous record {record['pid']}: Condition '{record['cond']}' linked to Plan '{record['med']}'."
    return "Error creating record."

In [ ]:
'''
Stage 2: Agent Orchestrator (The Brain)
What's an Agent? An AI that can decide which tools to use and in what order.
Framework: LangChain Agents or LangGraph
How It Works:

Receives user question
Analyzes what information is needed
Decides which tools to call
Combines results
Generates final answer
'''

# 1. Correct Imports
from langchain_ollama import ChatOllama
import google.genai
from langchain_core.tools import Tool
from google.colab import userdata
from langgraph.prebuilt import create_react_agent # The modern replacement
from langchain_core.messages import SystemMessage

# 2. Define Tools (Ensure vectorstore and create_patient_record are defined)
vsearch = Tool(
        name="VectorSearch",
        func=lambda q: vectorstore.similarity_search(q), # Wrapped in lambda for safety
        description="Search medical documents for relevant information"
    )
gquery = Tool(
        name="GraphQuery",
        func=create_patient_record,
        description="Useful for looking up medical records or patient conditions."
    )
tools = [vsearch, gquery]

# 3. Initialize Model

gemini_key = userdata.get('gemini_api_key')

# LLM
client = google.genai.Client(api_key=gemini_key)
# Start a chat session
llm = client.chats.create(model="gemini-2.5-flash")



# 4. Create the Agent (Replaces initialize_agent)
# We pass the system prompt as a state modifier
system_message = "You are a helpful medical assistant."
agent_executor = create_react_agent(llm, tools, prompt=system_message)

# 5. Run the Agent
# In LangGraph, we pass a dictionary with a "messages" list
query = "What medications is John taking and what are the side effects?"
response = agent_executor.invoke({"messages": [("user", query)]})

# Print the last message from the AI
print(response["messages"][-1].content)

# Behind the scenes:
# 1. Agent thinks: "I need patient info and medication info"
# 2. Calls GraphQuery tool → gets medications
# 3. Calls VectorSearch tool → gets side effects
# 4. Combines and answers

In [ ]:
'''
Stage 3: Specialized Tools
Tool 1: RAG Pipeline (Vector Search)
Purpose: Retrieve relevant document chunks for context
'''
from google.colab import userdata
import google.genai

# Initialize Model

gemini_key = userdata.get('gemini_api_key')

# LLM
client = google.genai.Client(api_key=gemini_key)
# Start a chat session
llm = client.chats.create(model="gemini-2.5-flash")

# Complete RAG example
def rag_pipeline(query: str):
    # 1. Embed the query
    query_vector = embeddings_model.embed_query(query)

    # 2. Find similar chunks
    relevant_chunks = vectorstore.similarity_search(query, k=3)

    # 3. Build context
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    # 4. Create prompt
    prompt = f"""
    Context: {context}

    Question: {query}

    Answer the question using only the context provided.
    """

    # 5. Get LLM response
    response = llm.send_message(prompt)

    return response

# Usage
answer = rag_pipeline("What is the recommended dosage for metformin?")